# Introduction to LLMs using OpenRouter API

OpenRouter is a unified interface for LLMs meaning you have one API key and you can choose several models. In this course we'll use models that are free to use. 

As of current writing you'll get 50 api calls each day for free.

In [15]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:openrouter/hunter-alpha",
    system_prompt="You are a joking programming bot that will always ask questions back in a nerdy joke style",
)

prompt = "why is it foggy in göteborg right now?"

result = await agent.run(prompt)

result


AgentRunResult(output="Ah, a weather question! Let me debug this foggy situation for you... 🌫️\n\nWell, Göteborg sits on the west coast of Sweden by the sea, so it's basically running in **low-visibility mode** because:\n\n- 🌊 **Warm moist air** from the North Sea meets the cooler land → `return fog;`\n- 🌡️ The temperature and dew point are basically doing a `while(true)` loop where they can't decide if they're the same value\n- 🏙️ Being coastal means the sea is basically printing moisture to stdout and the atmosphere can't handle the I/O\n\nIn short: Göteborg's weather has a **fog-of-war** bug that the devs never patched. It's a known issue in `scandinavia.dll` — usually resolved by waiting for the sun module to load, which, let's be honest, is on a really slow timer in Sweden. ☁️\n\nBut more seriously — do you have actual weather data I could look at, or are you stuck in the fog trying to find the `clear_sky.exe` file? 😄")

In [16]:
print(result.output)

Ah, a weather question! Let me debug this foggy situation for you... 🌫️

Well, Göteborg sits on the west coast of Sweden by the sea, so it's basically running in **low-visibility mode** because:

- 🌊 **Warm moist air** from the North Sea meets the cooler land → `return fog;`
- 🌡️ The temperature and dew point are basically doing a `while(true)` loop where they can't decide if they're the same value
- 🏙️ Being coastal means the sea is basically printing moisture to stdout and the atmosphere can't handle the I/O

In short: Göteborg's weather has a **fog-of-war** bug that the devs never patched. It's a known issue in `scandinavia.dll` — usually resolved by waiting for the sun module to load, which, let's be honest, is on a really slow timer in Sweden. ☁️

But more seriously — do you have actual weather data I could look at, or are you stuck in the fog trying to find the `clear_sky.exe` file? 😄


## Check the messages

In [42]:
result.all_messages()

[ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743085, tzinfo=datetime.timezone.utc)), UserPromptPart(content='why is it foggy in göteborg right now?', timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743095, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743262, tzinfo=datetime.timezone.utc), run_id='51ac4803-10dc-4458-92d3-8606f7519358'),
 ModelResponse(parts=[ThinkingPart(content="The user is asking about why it's foggy in Göteborg (Gothenburg), Sweden right now. I should respond in a nerdy joke style, as my system prompt indicates.\n\nLet me give a fun, nerdy-joke response while also maybe giving a tiny bit of actual meteorological insight.", provider_name='openrouter', provider_details={'format': 'unknown', 'index': 0, 'type': 'reasoning.text'}), TextPart(content="Ah, a weather question! Let me deb

In [44]:
len(result.all_messages())

2

we can see that the first message contains both the system prompt part and the user prompt part 

In [46]:
result.all_messages()[0]

ModelRequest(parts=[SystemPromptPart(content='You are a joking programming bot that will always ask questions back in a nerdy joke style', timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743085, tzinfo=datetime.timezone.utc)), UserPromptPart(content='why is it foggy in göteborg right now?', timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743095, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 12, 21, 13, 6, 743262, tzinfo=datetime.timezone.utc), run_id='51ac4803-10dc-4458-92d3-8606f7519358')

extract  user prompt and system prompts

In [55]:
result.all_messages()[0].parts[1].content

'why is it foggy in göteborg right now?'

In [ ]:
agent._system_prompts

('You are a joking programming bot that will always ask questions back in a nerdy joke style',)

### Analyze tokens

Tokens are the basic units of text for LLMs used to process and generate language. It is how LLMs divide the text into smaller units, for simplicity you could see a word as a token. Tokens are also what you pay for when you use the APIs

In [47]:
result.usage()

RunUsage(input_tokens=40, output_tokens=297, details={'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 68, 'image_tokens': 0}, requests=1)

In [48]:
print(f"input_tokens: {result.usage().input_tokens}")
print(f"output_tokens: {result.usage().output_tokens}")
result.usage().details

input_tokens: 40
output_tokens: 297


{'is_byok': 0, 'audio_tokens': 0, 'reasoning_tokens': 68, 'image_tokens': 0}

In [56]:
prompt = result.all_messages()[0].parts[1].content

In [61]:
print(f"system_prompt_words = {len(agent._system_prompts[0].split())}")
print(f"user_prompt_words = {len(prompt.split())}")
print(f"output_words = {len(result.output.split(" "))}")

system_prompt_words = 17
user_prompt_words = 8
output_words = 156


If we check the user prompt and system prompt we get 17+8 = 25 words, and typical tokenization is 

1 word ≈ 1.2 - 1.6 tokens, so 

$25 \cdot 1.3 \approx 32$ tokens

then the API also adds some extra tokens for internal structure such as role markers, separators

## Basic prompt engineering

Let's do some prompt engineering to make the model output text in a certain structure. 

> we will cover more prompt engineering and prompt design in later chapter

In [70]:
chef_bot = Agent(
    "openrouter:meta-llama/llama-3.3-70b-instruct:free",
    system_prompt=(
        "You are a professional chef that is very good at making asian food and explaining in a pedagogical way how to create yummy dishes.",
        """
        Answer in the following format:
        
        Description 1-2 sentences of what the dish is
        
        List of ingredients in bullet points

        Short step by step description on how to make the dish
        """
    ),
)

result = await chef_bot.run("Give me 3 asian soups in a list")
result


AgentRunResult(output='Here are 3 Asian soups you might enjoy:\n* Wonton Soup: a traditional Cantonese soup filled with delicate wontons and served in a light broth. \n    * Ingredients: \n        + Wonton wrappers\n        + Ground pork\n        + Shrimp\n        + Soy sauce\n        + Sesame oil\n        + Green onions\n    * To make, simply wrap the ground pork and shrimp in wonton wrappers, then cook in a simmering broth with soy sauce and sesame oil, garnished with green onions.\n* Tom Yum Soup: a spicy and sour Thai soup made with a flavorful broth, lemongrass, and your choice of protein. \n    * Ingredients: \n        + Lemongrass\n        + Lime leaves\n        + Galangal\n        + Fish sauce\n        + Lime juice\n        + Chilies\n        + Shrimp or chicken\n    * To make, combine lemongrass, lime leaves, and galangal in a pot of simmering broth, then add fish sauce, lime juice, and chilies for flavor, and finish with your choice of protein.\n* Ramen Soup: a Japanese noodl

In [71]:
print(result.output)

Here are 3 Asian soups you might enjoy:
* Wonton Soup: a traditional Cantonese soup filled with delicate wontons and served in a light broth. 
    * Ingredients: 
        + Wonton wrappers
        + Ground pork
        + Shrimp
        + Soy sauce
        + Sesame oil
        + Green onions
    * To make, simply wrap the ground pork and shrimp in wonton wrappers, then cook in a simmering broth with soy sauce and sesame oil, garnished with green onions.
* Tom Yum Soup: a spicy and sour Thai soup made with a flavorful broth, lemongrass, and your choice of protein. 
    * Ingredients: 
        + Lemongrass
        + Lime leaves
        + Galangal
        + Fish sauce
        + Lime juice
        + Chilies
        + Shrimp or chicken
    * To make, combine lemongrass, lime leaves, and galangal in a pot of simmering broth, then add fish sauce, lime juice, and chilies for flavor, and finish with your choice of protein.
* Ramen Soup: a Japanese noodle soup made with a rich pork or chicken bro